# Session 13 — Missing Values, Duplicates, Data Cleaning Strategy
### Module 4: Data Cleaning, Visualization and EDA

This notebook covers:
- Identifying missing values in a real, messy dataset
- Understanding *why* data goes missing and what that means for strategy
- Column-by-column cleaning strategy (with reasoning in comments)
- Identifying and removing duplicate records
- General-purpose cleaning techniques: drop, fill, impute, flag
- Hands-on practice exercises

We will use the **Titanic dataset** — a real, genuinely messy dataset with missing values across multiple columns of different types (numeric, categorical, mostly-empty). This is a different dataset from the ones used in earlier sessions (not wine, not iris).


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
'''| Column          | Data Type        | Description                                                             | Example              |
| --------------- | ---------------- | ----------------------------------------------------------------------- | -------------------- |
| **survived**    | Integer (Target) | Whether the passenger survived. **0 = No, 1 = Yes**                     | 1                    |
| **pclass**      | Integer          | Passenger ticket class. **1 = First, 2 = Second, 3 = Third**            | 1                    |
| **sex**         | Categorical      | Gender of the passenger.                                                | male, female         |
| **age**         | Numeric          | Age of the passenger in years.                                          | 22                   |
| **sibsp**       | Integer          | Number of **siblings/spouses** traveling with the passenger.            | 1                    |
| **parch**       | Integer          | Number of **parents/children** traveling with the passenger.            | 0                    |
| **fare**        | Numeric          | Ticket fare paid by the passenger.                                      | 72.50                |
| **embarked**    | Categorical      | Port where the passenger boarded the ship.                              | C, Q, S              |
| **class**       | Categorical      | Passenger class in text format.                                         | First, Second, Third |
| **who**         | Categorical      | Passenger category based on age and gender.                             | man, woman, child    |
| **adult_male**  | Boolean          | Whether the passenger is an adult male.                                 | True                 |
| **deck**        | Categorical      | Deck (cabin level) where the passenger stayed. Many values are missing. | A, B, C              |
| **embark_town** | Categorical      | Full name of the embarkation town.                                      | Southampton          |
| **alive**       | Categorical      | Survival status in text.                                                | yes, no              |
| **alone**       | Boolean          | Whether the passenger was traveling alone.                              | True                 |
'''

'| Column          | Data Type        | Description                                                             | Example              |\n| --------------- | ---------------- | ----------------------------------------------------------------------- | -------------------- |\n| **survived**    | Integer (Target) | Whether the passenger survived. **0 = No, 1 = Yes**                     | 1                    |\n| **pclass**      | Integer          | Passenger ticket class. **1 = First, 2 = Second, 3 = Third**            | 1                    |\n| **sex**         | Categorical      | Gender of the passenger.                                                | male, female         |\n| **age**         | Numeric          | Age of the passenger in years.                                          | 22                   |\n| **sibsp**       | Integer          | Number of **siblings/spouses** traveling with the passenger.            | 1                    |\n| **parch**       | Integer          | N

## 1. Loading the Dataset

`seaborn` ships with several sample datasets, including `titanic`, which can be loaded directly without needing a local file.


In [3]:
df = sns.load_dataset('titanic')
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [4]:
df.shape

(891, 15)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [43]:
df['embark_town'].value_counts()

embark_town
Southampton    649
Cherbourg      170
Queenstown      77
Name: count, dtype: int64

**About the dataset:** each row is a passenger on the Titanic. Columns include a mix of numeric (`age`, `fare`), categorical (`sex`, `embarked`, `class`), and boolean/derived columns (`alone`, `adult_male`). This is genuinely messy — several columns have real missing values, which is exactly what we need for this session.


## 2. Simulating Duplicate Records

The raw Titanic dataset does not contain exact duplicate rows, so for practice purposes we will deliberately duplicate a few rows. **In a real project you would NOT do this step** — it is only done here so we have duplicates to detect and remove later in this notebook.


In [6]:
# Deliberately injecting duplicate rows for practice purposes only
duplicate_rows = df.sample(5, random_state=42)
df = pd.concat([df, duplicate_rows], ignore_index=True)
print("New shape after injecting duplicates:", df.shape)

New shape after injecting duplicates: (896, 15)


## 3. Identifying Missing Values


In [7]:
df.isnull().sum()          # count of missing values per column

survived         0
pclass           0
sex              0
age            178
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           693
embark_town      2
alive            0
alone            0
dtype: int64

In [8]:
#df.isnull().sum()   
df.isnull().mean() * 100   # percentage of missing values per column

survived        0.000000
pclass          0.000000
sex             0.000000
age            19.866071
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.223214
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.343750
embark_town     0.223214
alive           0.000000
alone           0.000000
dtype: float64

In [9]:
df.isnull().sum().sort_values(ascending=False)   # sorted, easiest to read

deck           693
age            178
embarked         2
embark_town      2
survived         0
pclass           0
sex              0
sibsp            0
parch            0
fare             0
class            0
who              0
adult_male       0
alive            0
alone            0
dtype: int64

## 4. Understanding Why Data Goes Missing

Before deciding a strategy, always ask **why** a value might be missing:

| Type | Meaning | Example |
|---|---|---|
| MCAR (Missing Completely at Random) | No pattern, pure chance | A sensor randomly failed to log a reading |
| MAR (Missing at Random) | Missingness relates to another column | Older records more likely missing `age` |
| MNAR (Missing Not at Random) | Missingness relates to the value itself | People with low income skip the "income" field |

This context should guide whether you drop, fill, or flag a column — not just the percentage missing.


## 5. Column-by-Column Cleaning Strategy

Below, each column with missing values is handled individually with the reasoning written as a comment. This is the recommended approach — a single blanket strategy (like "fill everything with 0") is rarely correct.


In [10]:
print("Before:", df['age'].isnull().sum())

Before: 178


In [11]:
# ---- age ----
# Numeric column, ~20% missing.
# Strategy: impute using median grouped by 'pclass' and 'sex', since age tends to
# vary by passenger class and gender. Median is preferred over mean because age
# can be skewed by outliers (very young infants, very old passengers).
df['age'] = df.groupby(['pclass', 'sex'])['age'].transform(lambda x: x.fillna(x.median()))

# Fallback: if any group had no data at all, use the overall median
df['age'] = df['age'].fillna(df['age'].median())

In [12]:
print("After:", df['age'].isnull().sum())

After: 0


In [13]:
df['embarked'].isnull().sum()

2

In [44]:
df['embarked'].mode()[0]

'S'

In [14]:
# ---- embarked ----
# Categorical column, only 2 missing values (a very small fraction).
# Strategy: impute with the mode (most frequent value), since dropping 2 rows
# out of ~890 loses very little information, but imputing keeps the dataset intact.
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

In [15]:
df['embarked'].isnull().sum()

0

In [16]:
df['embark_town'].isnull().sum()

2

In [17]:
# ---- embark_town ----
# Same information as 'embarked', just a different format (full town name).
# Strategy: impute using the same logic/mode, OR derive it directly from the
# now-cleaned 'embarked' column to guarantee consistency between the two.
embarked_to_town = {'S': 'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'}
df['embark_town'] = df['embark_town'].fillna(df['embarked'].map(embarked_to_town))

In [18]:
df['embark_town'].isnull().sum()

0

In [19]:
# ---- deck ----
# Categorical column, majority missing (~77%).
# Strategy: too much missing data to reliably impute. Instead of dropping the
# column entirely (it may still carry signal), create a binary indicator column
# capturing whether deck information was recorded at all, then drop the original.
df['deck_known'] = df['deck'].notnull().astype(int)
df = df.drop(columns=['deck'])

In [45]:
df['deck_known'].value_counts()

deck_known
0    693
1    203
Name: count, dtype: int64

In [20]:
df['fare'].isnull().sum()

0

In [21]:
# ---- fare ----
# Numeric column, may have 0 or very few missing values in this dataset,
# but handled here defensively in case of unseen/future data.
# Strategy: impute with median (robust to outliers e.g. very high first-class fares).
df['fare'] = df['fare'].fillna(df['fare'].median())


In [22]:
df['fare'].isnull().sum()

0

In [23]:
# Re-check missing values after all column-level cleaning
df.isnull().sum()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
deck_known     0
dtype: int64

## 6. General-Purpose Missing Value Techniques

Beyond column-specific logic, these are the standard tools available:


In [24]:
# Drop rows with ANY missing values (aggressive - can lose a lot of data)
df_drop_rows = df.dropna()
print(df_drop_rows.shape)

(896, 15)


In [25]:
# Drop rows with missing values only in specific columns
df_drop_subset = df.dropna(subset=['embarked'])
print(df_drop_subset.shape)

(896, 15)


In [26]:
# Drop columns where more than a threshold percentage is missing
threshold = 0.5   # 50%
df_drop_cols = df.loc[:, df.isnull().mean() < threshold]
print(df_drop_cols.shape)


(896, 15)


In [27]:
# Fill with a constant value
df['who'] = df['who'].fillna('unknown')


In [28]:
# Forward fill / backward fill (useful for time-ordered data)
df_ffill = df.fillna(method='ffill')
df_bfill = df.fillna(method='bfill')


C:\Users\DELL\AppData\Local\Temp\ipykernel_18284\3952085057.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ffill = df.fillna(method='ffill')
C:\Users\DELL\AppData\Local\Temp\ipykernel_18284\3952085057.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_bfill = df.fillna(method='bfill')


In [29]:
# Interpolation (useful for numeric, sequential data like time series)
df_interp = df.copy()
df_interp['age'] = df_interp['age'].interpolate()


**Decision guide:**

| Situation | Recommended approach |
|---|---|
| Very few rows missing (< ~5%) | Keep rows with a approach |
| Column mostly missing (> ~50-60%) | Drop column, or keep a "was it recorded" indicator |
| Numeric column, moderate missing | Impute with mean/median (median if skewed) |
| Categorical column, moderate missing | Impute with mode, or an explicit "Unknown" category |
| Time-ordered / sequential data | Forward-fill, backward-fill, or interpolate |
| Missingness itself may be meaningful | Create a binary indicator column before dropping/filling |


## 7. Identifying and Removing Duplicates


In [30]:
df.duplicated().sum()          # count of fully duplicated rows

119

In [31]:
df[df.duplicated()]            # view the duplicate rows themselves

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone,deck_known
47,1,3,female,21.0,0,0,7.7500,Q,Third,woman,False,Queenstown,yes,True,0
64,0,1,male,40.0,0,0,27.7208,C,First,man,True,Cherbourg,no,True,0
76,0,3,male,25.0,0,0,7.8958,S,Third,man,True,Southampton,no,True,0
77,0,3,male,25.0,0,0,8.0500,S,Third,man,True,Southampton,no,True,0
87,0,3,male,25.0,0,0,8.0500,S,Third,man,True,Southampton,no,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
891,1,3,male,25.0,1,1,15.2458,C,Third,man,True,Cherbourg,yes,False,0
892,0,2,male,31.0,0,0,10.5000,S,Second,man,True,Southampton,no,True,0
893,0,3,male,20.0,0,0,7.9250,S,Third,man,True,Southampton,no,True,0
894,1,2,female,6.0,0,1,33.0000,S,Second,child,False,Southampton,yes,False,0


In [32]:
df[df.duplicated(keep=False)]  # show ALL copies involved in a duplicate (not just the extra one)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone,deck_known
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True,0
26,0,3,male,25.0,0,0,7.2250,C,Third,man,True,Cherbourg,no,True,0
28,1,3,female,21.0,0,0,7.8792,Q,Third,woman,False,Queenstown,yes,True,0
29,0,3,male,25.0,0,0,7.8958,S,Third,man,True,Southampton,no,True,0
30,0,1,male,40.0,0,0,27.7208,C,First,man,True,Cherbourg,no,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
891,1,3,male,25.0,1,1,15.2458,C,Third,man,True,Cherbourg,yes,False,0
892,0,2,male,31.0,0,0,10.5000,S,Second,man,True,Southampton,no,True,0
893,0,3,male,20.0,0,0,7.9250,S,Third,man,True,Southampton,no,True,0
894,1,2,female,6.0,0,1,33.0000,S,Second,child,False,Southampton,yes,False,0


In [33]:
# Remove duplicates, keeping the first occurrence (default behaviour)
df_clean = df.drop_duplicates()
print("Before:", df.shape, " After:", df_clean.shape)


Before: (896, 15)  After: (777, 15)


In [34]:
# Remove duplicates based on a subset of columns only
# (useful when two rows are "the same person" even if minor columns differ)
df_clean_subset = df.drop_duplicates(subset=['sex', 'age', 'fare', 'pclass'])
print(df_clean_subset.shape)


(740, 15)


In [35]:
# Keep the LAST occurrence instead of the first
df_clean_last = df.drop_duplicates(keep='last')


## 8. Final Cleaned Dataset


In [36]:
df_final = df.drop_duplicates().reset_index(drop=True)
df_final.isnull().sum()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
deck_known     0
dtype: int64

In [37]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 777 entries, 0 to 776
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     777 non-null    int64   
 1   pclass       777 non-null    int64   
 2   sex          777 non-null    object  
 3   age          777 non-null    float64 
 4   sibsp        777 non-null    int64   
 5   parch        777 non-null    int64   
 6   fare         777 non-null    float64 
 7   embarked     777 non-null    object  
 8   class        777 non-null    category
 9   who          777 non-null    object  
 10  adult_male   777 non-null    bool    
 11  embark_town  777 non-null    object  
 12  alive        777 non-null    object  
 13  alone        777 non-null    bool    
 14  deck_known   777 non-null    int32   
dtypes: bool(2), category(1), float64(2), int32(1), int64(4), object(5)
memory usage: 72.3+ KB


In [38]:
df_final.to_csv('titanic_cleaned.csv', index=False)
print("Cleaned dataset saved as titanic_cleaned.csv")


Cleaned dataset saved as titanic_cleaned.csv


## 9. Overall Data Cleaning Strategy — Summary

1. **Inspect first** — always run `.info()`, `.isnull().sum()`, and `.duplicated().sum()` before touching anything
2. **Understand each column individually** — a numeric column, a categorical column, and a mostly-empty column all need different treatment
3. **Prefer targeted fixes over blanket rules** — grouped medians, mode imputation, and indicator columns preserve more information than dropping everything
4. **Handle duplicates deliberately** — decide whether "duplicate" means an identical row, or identical on a meaningful subset of columns
5. **Re-check after cleaning** — confirm `.isnull().sum()` and `.duplicated().sum()` are at the expected state before moving to analysis
6. **Document your reasoning** — as done in the comments above, so anyone reviewing the notebook understands *why* a strategy was chosen, not just what was done


## 10. Hands-on Practice Exercise 1 — Missing Values

Using `df` (before final cleaning, reload with `sns.load_dataset('titanic')` if needed):

1. Find which column has the highest percentage of missing values
2. Impute `age` using the overall median (not grouped) and compare the result to the grouped-median approach used above
3. Drop all columns with more than 60% missing values
4. Explain in a comment why you would or would not drop the `deck` column entirely instead of keeping an indicator


In [39]:
# Write your solution here



## 11. Hands-on Practice Exercise 2 — Duplicates

1. Reload a fresh copy of the Titanic dataset and confirm it has zero duplicates
2. Manually create 3 duplicate rows (as done in Section 2) and confirm `.duplicated().sum()` detects them
3. Remove duplicates keeping the **last** occurrence instead of the first
4. Try `drop_duplicates(subset=...)` using only 2 columns of your choice — how many rows get removed compared to using all columns?


In [40]:
# Write your solution here



## 12. Hands-on Practice Exercise 3 — End-to-End Cleaning

1. Reload a fresh copy of the Titanic dataset
2. Write a full cleaning strategy: for every column with missing data, decide and apply drop / impute / flag, with a comment explaining your reasoning
3. Remove any duplicate records
4. Save the final cleaned dataset to a CSV file
5. Print the final `.info()` to confirm no missing values remain


In [41]:
# Write your solution here



## Key Terms Glossary

| Term | Meaning |
|---|---|
| `isnull()` / `isna()` | Detect missing values |
| MCAR / MAR / MNAR | Categories describing *why* data is missing |
| `dropna()` | Remove rows or columns with missing values |
| `fillna()` | Fill missing values with a constant, mean, median, mode, etc. |
| `groupby().transform()` | Fill missing values using group-specific statistics |
| Indicator column | A flag column marking whether a value was originally missing |
| `duplicated()` | Detect duplicate rows |
| `drop_duplicates()` | Remove duplicate rows |
| `subset=` | Restrict duplicate/missing-value checks to specific columns |
| `interpolate()` | Estimate missing numeric values based on surrounding data |


## Common Mistakes to Avoid

- Filling all missing values with a single blanket method (e.g. `fillna(0)` everywhere) regardless of column meaning
- Using mean imputation on a skewed column instead of median
- Dropping rows so aggressively that a large portion of the dataset is lost
- Forgetting to re-check `.isnull().sum()` and `.duplicated().sum()` after cleaning
- Treating "duplicate" only as an exact full-row match, when in real data duplicates often differ slightly (e.g. typos, re-entries) and need `subset=` logic
- Deleting a mostly-missing column without considering whether an indicator column would preserve useful signal


## Homework / Practice Assignment

1. Pick any dataset (can be `sns.load_dataset('penguins')`, `sns.load_dataset('tips')`, or your own CSV file) that contains real missing values
2. For every column with missing data, write out a short strategy comment (drop / impute / flag) and justify it
3. Check for and remove duplicates, trying both a full-row check and a subset-based check
4. Save the cleaned dataset as a new CSV file
5. Come prepared with 2 questions/doubts for the next session on outlier identification and treatment
